# **encoder with English**

In [1]:
!pip install sentencepiece
!pip install seqeval
!pip install tqdm
!pip install torch transformers

import json
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
from transformers import XLMRobertaTokenizerFast, XLMRobertaModel
import numpy as np
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import classification_report as sklearn_classification_report, accuracy_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=a0874176958b5c00c779f73013afd79d2966021d12ccee6b6f3948cf214dedd1
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


# **BIOایجاد توکن‌ها و برچسب‌ها با استفاده از برچسب گذاری**

In [3]:
def create_tokens_and_labels(id, sample):
    intent = sample['intent']
    utt = sample['utt']
    annot_utt = sample['annot_utt']
    tokens = utt.split()
    labels = []
    label = 'O'
    split_annot_utt = annot_utt.split()
    idx = 0
    BIO_SLOT = False
    while idx < len(split_annot_utt):
        if split_annot_utt[idx].startswith('['):
            label = split_annot_utt[idx].lstrip('[')
            idx += 2
            BIO_SLOT = True
        elif split_annot_utt[idx].endswith(']'):
            if split_annot_utt[idx-1] ==":":
                labels.append("B-"+label)
                label = 'O'
                idx += 1
            else:
                labels.append("I-"+label)
                label = 'O'
                idx += 1
            BIO_SLOT = False
        else:
            if split_annot_utt[idx-1] ==":":
                labels.append("B-"+label)
                idx += 1
            elif BIO_SLOT == True:
                labels.append("I-"+label)
                idx += 1
            else:
                labels.append("O")
                idx += 1

    if len(tokens) != len(labels):
        raise ValueError(f"Length of tokens, {tokens}, doesn't match length of labels, {labels}, "
                          f"for id {id} and annot_utt: {annot_utt}")
    return tokens, labels, intent

sentences_tr, tags_tr, intent_tags_tr = [], [], []
sentences_val, tags_val, intent_tags_val = [], [], []
sentences_test, tags_test, intent_tags_test = [], [], []

massive_raw = []
with open('/content/drive/MyDrive/en-US.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        massive_raw.append(json.loads(line))

for id, sample in enumerate(massive_raw):
    tokens, labels, intent = create_tokens_and_labels(id, sample)
    if sample['partition'] == 'train':
        sentences_tr.append(tokens)
        tags_tr.append(labels)
        intent_tags_tr.append(intent)
    elif sample['partition'] == 'dev':
        sentences_val.append(tokens)
        tags_val.append(labels)
        intent_tags_val.append(intent)
    elif sample['partition'] == 'test':
        sentences_test.append(tokens)
        tags_test.append(labels)
        intent_tags_test.append(intent)

for i in range(5):
    print(f"Sentence {i+1}: {sentences_tr[i]}")
    print(f"Labels {i+1}: {tags_tr[i]}")
    print(f"Intent {i+1}: {intent_tags_tr[i]}\n")

print("Development Set Samples:")
for i in range(min(5, len(sentences_val))):
    print(f"Sentence: {sentences_val[i]}")
    print(f"Labels: {tags_val[i]}")
    print(f"Intent: {intent_tags_val[i]}\n")

print("Test Set Samples:")
for i in range(min(5, len(sentences_test))):
    print(f"Sentence: {sentences_test[i]}")
    print(f"Labels: {tags_test[i]}")
    print(f"Intent: {intent_tags_test[i]}\n")

print("First few samples from the dataset:")
for i in range(1):
    print(massive_raw[i])


partition_counts = Counter(sample['partition'] for sample in massive_raw)
print("Partition counts:")
for partition, count in sorted(partition_counts.items()):
    print(f"  {partition.capitalize()}: {count}")


Sentence 1: ['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday']
Labels 1: ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']
Intent 1: alarm_set

Sentence 2: ['set', 'an', 'alarm', 'for', 'two', 'hours', 'from', 'now']
Labels 2: ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'I-time', 'I-time']
Intent 2: alarm_set

Sentence 3: ['olly', 'quiet']
Labels 3: ['O', 'O']
Intent 3: audio_volume_mute

Sentence 4: ['stop']
Labels 4: ['O']
Intent 4: audio_volume_mute

Sentence 5: ['olly', 'pause', 'for', 'ten', 'seconds']
Labels 5: ['O', 'O', 'O', 'B-time', 'I-time']
Intent 5: audio_volume_mute

Development Set Samples:
Sentence: ['turn', 'the', 'lights', 'off', 'please']
Labels: ['O', 'O', 'O', 'O', 'O']
Intent: iot_hue_lightoff

Sentence: ['dim', 'the', 'lights', 'in', 'the', 'hall']
Labels: ['O', 'O', 'O', 'O', 'O', 'B-house_place']
Intent: iot_hue_lightdim

Sentence: ['make', 'a', 'room', 'darker']
Labels: ['O', 'O', 'B-house_place', 'O']
Intent: iot_hue_lightdim

Sentence: ['clean', 'th

# **intent , slot شمارش تعداد برچسب‌های یکتا و مپ کردن**

In [4]:
all_slot_labels = tags_tr + tags_val + tags_test
all_intent_labels = intent_tags_tr + intent_tags_val + intent_tags_test

unique_slot_labels = set([label for sublist in all_slot_labels for label in sublist])
unique_intent_labels = set(all_intent_labels)

num_intent_labels = len(unique_intent_labels)
num_slot_labels = len(unique_slot_labels)
print("Number of unique slot labels:", num_slot_labels)
print("Number of unique intent labels:", num_intent_labels)

# ایجاد مپینگ برای برچسب‌های اسلات
slot_label_to_id = {label: i for i, label in enumerate(unique_slot_labels)}
id_to_slot_label = {i: label for label, i in slot_label_to_id.items()}

# ایجاد مپینگ برای برچسب‌های نیت
intent_label_to_id = {label: i for i, label in enumerate(unique_intent_labels)}
id_to_intent_label = {i: label for label, i in intent_label_to_id.items()}


Number of unique slot labels: 108
Number of unique intent labels: 60


#  xlm-roberta-baseپیش پردازش داده‌ها و توکنایز با

In [5]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')
def tokenize_and_align_labels(sentences, labels, tokenizer):
    tokenized_inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt", is_split_into_words=True)
    labels_aligned = []

    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(slot_label_to_id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels_aligned.append(label_ids)

    return tokenized_inputs, labels_aligned

tokenized_inputs_tr, labels_aligned_tr = tokenize_and_align_labels(sentences_tr, tags_tr, tokenizer)
tokenized_inputs_val, labels_aligned_val = tokenize_and_align_labels(sentences_val, tags_val, tokenizer)
tokenized_inputs_test, labels_aligned_test = tokenize_and_align_labels(sentences_test, tags_test, tokenizer)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

In [6]:
class IntentSlotDataset(Dataset):
    def __init__(self, tokenized_data, labels, intent_labels):
        self.tokenized_data = tokenized_data
        self.labels = labels
        self.intent_labels = intent_labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.tokenized_data.items() if key != 'offset_mapping'}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['intent_labels'] = torch.tensor(self.intent_labels[idx], dtype=torch.long)
        return item

numeric_intent_labels_tr = [intent_label_to_id[label] for label in intent_tags_tr]
numeric_intent_labels_val = [intent_label_to_id[label] for label in intent_tags_val]
numeric_intent_labels_test = [intent_label_to_id[label] for label in intent_tags_test]

train_dataset = IntentSlotDataset(tokenized_inputs_tr, labels_aligned_tr, numeric_intent_labels_tr)
val_dataset = IntentSlotDataset(tokenized_inputs_val, labels_aligned_val, numeric_intent_labels_val)
test_dataset = IntentSlotDataset(tokenized_inputs_test, labels_aligned_test, numeric_intent_labels_test)


#  xlm-roberta-baseمعماری مدل

In [7]:
class IntentSlotModel(nn.Module):
    def __init__(self, num_slot_labels, num_intent_labels, dropout_rate=0.1):
        super(IntentSlotModel, self).__init__()
        self.xlm_roberta = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.dropout = nn.Dropout(dropout_rate)
        self.slot_classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_slot_labels)
        self.intent_classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_intent_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]

        sequence_output = self.dropout(sequence_output)

        slot_logits = self.slot_classifier(sequence_output)

        # طبقه‌بند نیت (استفاده از خروجی توکن اول)
        intent_logits = self.intent_classifier(sequence_output[:, 0, :])

        return slot_logits, intent_logits

num_slot_labels = len(unique_slot_labels)
num_intent_labels = len(unique_intent_labels)
model = IntentSlotModel(num_slot_labels, num_intent_labels)


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [8]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=2.8e-5)
criterion = nn.CrossEntropyLoss(ignore_index=-100)


**آموزش مدل با ۲۰ ایپاک**

In [10]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    total_train_intent_loss = 0
    total_train_slot_loss = 0
    all_intent_preds, all_intent_targets = [], []
    all_slot_preds, all_slot_targets = [], []

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        slot_labels = batch['labels'].to(device)
        intent_labels = batch['intent_labels'].to(device)

        optimizer.zero_grad()
        slot_logits, intent_logits = model(input_ids, attention_mask)

        slot_loss = criterion(slot_logits.view(-1, num_slot_labels), slot_labels.view(-1))
        intent_loss = criterion(intent_logits, intent_labels)
        loss = slot_loss + intent_loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        total_train_intent_loss += intent_loss.item()
        total_train_slot_loss += slot_loss.item()

        intent_preds = torch.argmax(intent_logits, dim=1)
        slot_preds = torch.argmax(slot_logits, dim=2)

        all_intent_preds.extend(intent_preds.cpu().numpy())
        all_intent_targets.extend(intent_labels.cpu().numpy())

    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_intent_loss = total_train_intent_loss / len(train_loader)
    avg_train_slot_loss = total_train_slot_loss / len(train_loader)
    train_intent_accuracy = accuracy_score(all_intent_targets, all_intent_preds)

    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Intent Loss: {avg_train_intent_loss:.4f}, Slot Loss: {avg_train_slot_loss:.4f}, Train Intent Acc: {train_intent_accuracy:.4f}")


Epoch 1/20, Train Loss: 3.7487, Intent Loss: 2.6221, Slot Loss: 1.1266, Train Intent Acc: 0.3828
Epoch 2/20, Train Loss: 1.4382, Intent Loss: 0.8538, Slot Loss: 0.5844, Train Intent Acc: 0.8015
Epoch 3/20, Train Loss: 0.9250, Intent Loss: 0.5100, Slot Loss: 0.4150, Train Intent Acc: 0.8782
Epoch 4/20, Train Loss: 0.6856, Intent Loss: 0.3669, Slot Loss: 0.3187, Train Intent Acc: 0.9097
Epoch 5/20, Train Loss: 0.5291, Intent Loss: 0.2674, Slot Loss: 0.2618, Train Intent Acc: 0.9339
Epoch 6/20, Train Loss: 0.4346, Intent Loss: 0.2100, Slot Loss: 0.2246, Train Intent Acc: 0.9508
Epoch 7/20, Train Loss: 0.3460, Intent Loss: 0.1586, Slot Loss: 0.1875, Train Intent Acc: 0.9607
Epoch 8/20, Train Loss: 0.2937, Intent Loss: 0.1253, Slot Loss: 0.1684, Train Intent Acc: 0.9679
Epoch 9/20, Train Loss: 0.2440, Intent Loss: 0.0973, Slot Loss: 0.1467, Train Intent Acc: 0.9765
Epoch 10/20, Train Loss: 0.2191, Intent Loss: 0.0883, Slot Loss: 0.1308, Train Intent Acc: 0.9766
Epoch 11/20, Train Loss: 0.18

In [11]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Data_slot/intent_slot_model_EN.pt")


In [16]:
def align_predictions(predictions, label_ids):
    preds = np.argmax(predictions, axis=2)
    batch_size = preds.shape[0]
    labels_list, preds_list = [], []

    for batch_index in range(batch_size):
        batch_label_ids = label_ids[batch_index]
        batch_preds = preds[batch_index]

        filtered_labels = [id_to_slot_label[label_id] for label_id in batch_label_ids if label_id != -100]
        filtered_preds = [id_to_slot_label[pred] for pred, label_id in zip(batch_preds, batch_label_ids) if label_id != -100]

        labels_list.append(filtered_labels)
        preds_list.append(filtered_preds)

    return preds_list, labels_list


In [17]:
def evaluate_model(model, dataset, description):
    model.eval()
    eval_loss = 0
    intent_preds, intent_true = [], []
    slot_preds, slot_true = [], []

    with torch.no_grad():
        for batch in DataLoader(dataset, batch_size=32):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            slot_labels = batch['labels'].to(device)
            intent_labels = batch['intent_labels'].to(device)

            slot_logits, intent_logits = model(input_ids, attention_mask)

            slot_loss = criterion(slot_logits.view(-1, num_slot_labels), slot_labels.view(-1))
            intent_loss = criterion(intent_logits, intent_labels)
            loss = slot_loss + intent_loss

            eval_loss += loss.item()

            # پیش‌بینی نیت‌ها
            intent_preds.extend(torch.argmax(intent_logits, dim=1).cpu().numpy())
            intent_true.extend(intent_labels.cpu().numpy())

            # پیش‌بینی اسلات‌ها
            batch_slot_preds, batch_slot_true = align_predictions(slot_logits.detach().cpu().numpy(), slot_labels.detach().cpu().numpy())
            slot_preds.extend(batch_slot_preds)
            slot_true.extend(batch_slot_true)

    avg_loss = eval_loss / len(DataLoader(dataset, batch_size=32))
    intent_accuracy = accuracy_score(intent_true, intent_preds)
    unique_intent_ids = sorted(set(intent_true))
    target_names = [id_to_intent_label[intent_id] for intent_id in unique_intent_ids]

    intent_report = sklearn_classification_report(intent_true, intent_preds, labels=unique_intent_ids, target_names=target_names)
    print(f"{description} Intent Classification Report:\n{intent_report}")

    slot_precision_micro = precision_score(slot_true, slot_preds, average='micro')
    slot_recall_micro = recall_score(slot_true, slot_preds, average='micro')
    slot_f1_micro = f1_score(slot_true, slot_preds, average='micro')

    slot_precision_macro = precision_score(slot_true, slot_preds, average='macro')
    slot_recall_macro = recall_score(slot_true, slot_preds, average='macro')
    slot_f1_macro = f1_score(slot_true, slot_preds, average='macro')

    print(f"{description} Slot Filling Metrics:")
    print(f"Micro Precision: {slot_precision_micro:.4f}, Micro Recall: {slot_recall_micro:.4f}, Micro F1: {slot_f1_micro:.4f}")
    print(f"Macro Precision: {slot_precision_macro:.4f}, Macro Recall: {slot_recall_macro:.4f}, Macro F1: {slot_f1_macro:.4f}")

    slot_detailed_report = classification_report(slot_true, slot_preds)
    print(f"{description} Slot Filling Detailed Report:\n{slot_detailed_report}")


# **ارزیابی مدل روی داده‌های اعتبارسنجی**

In [18]:
evaluate_model(model, val_dataset, "Validation")




Validation Intent Classification Report:
                          precision    recall  f1-score   support

            lists_remove       0.97      0.92      0.94        37
             music_query       0.93      0.87      0.90        30
         email_sendemail       0.98      0.90      0.94        63
          music_settings       0.70      0.88      0.78         8
           qa_definition       0.89      0.89      0.89        55
       audio_volume_down       1.00      1.00      1.00         8
          calendar_query       0.88      0.85      0.87       102
             qa_currency       0.97      0.97      0.97        32
              news_query       0.78      0.91      0.84        82
             alarm_query       0.88      0.79      0.83        19
               alarm_set       0.83      0.94      0.88        31
        iot_hue_lightoff       0.84      0.94      0.89        17
          play_audiobook       0.91      0.86      0.88        35
            social_query       0.8

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0

Validation Slot Filling Metrics:
Micro Precision: 0.7790, Micro Recall: 0.8479, Micro F1: 0.8120
Macro Precision: 0.5961, Macro Recall: 0.6717, Macro F1: 0.6254
Validation Slot Filling Detailed Report:
                      precision    recall  f1-score   support

          alarm_type       0.00      0.00      0.00         2
            app_name       0.80      0.50      0.62         8
         artist_name       0.69      0.78      0.73        45
    audiobook_author       0.00      0.00      0.00         1
      audiobook_name       0.80      0.75      0.77        16
       business_name       0.90      0.90      0.90        58
       business_type       0.77      0.82      0.79        33
       change_amount       0.80      0.67      0.73         6
         coffee_type       0.50      0.50      0.50         2
          color_type       0.94      0.94      0.94        16
        cooking_type       0.00      0.00      0.00         2
       currency_name       0.84      0.96      0.90  

# **ارزیابی مدل روی داده‌های تست**

In [19]:
evaluate_model(model, test_dataset, "Test")

Test Intent Classification Report:
                          precision    recall  f1-score   support

            lists_remove       0.92      0.94      0.93        52
             music_query       0.89      0.89      0.89        35
         email_sendemail       0.97      0.94      0.96       114
          music_settings       0.67      0.33      0.44         6
           qa_definition       0.96      0.84      0.90        57
       audio_volume_down       0.92      1.00      0.96        11
          calendar_query       0.85      0.85      0.85       126
             qa_currency       0.97      0.95      0.96        39
              news_query       0.84      0.91      0.88       124
             alarm_query       1.00      1.00      1.00        34
               alarm_set       0.93      0.98      0.95        41
        iot_hue_lightoff       1.00      0.93      0.96        43
          play_audiobook       0.82      0.80      0.81        41
            social_query       0.88     

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Test Slot Filling Metrics:
Micro Precision: 0.7600, Micro Recall: 0.8188, Micro F1: 0.7883
Macro Precision: 0.6288, Macro Recall: 0.6863, Macro F1: 0.6485
Test Slot Filling Detailed Report:
                      precision    recall  f1-score   support

          alarm_type       0.67      0.67      0.67         3
            app_name       0.50      0.60      0.55         5
         artist_name       0.67      0.84      0.74        61
    audiobook_author       0.33      0.20      0.25         5
      audiobook_name       0.76      0.70      0.73        23
       business_name       0.78      0.75      0.76        92
       business_type       0.65      0.69      0.67        32
       change_amount       0.70      0.88      0.78         8
         coffee_type       0.67      1.00      0.80         4
          color_type       0.79      0.88      0.84        26
        cooking_type       1.00      0.62      0.77         8
       currency_name       0.83      0.88      0.85        50
   

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
